# Day 1 · 03 · Kimball Model Review & Power BI Preview

## Business Scenario

Workshop 1 produced the first durable Gold star schema. This short closing demo
turns the objects into a BI handoff: what Power BI should connect to, which
columns support relationships, which checks prove the grain and what must be
finished before Day 2.


## Learning Objectives

By the end of this review participants can:

- explain what was built in Workshop 1,
- validate the Gold fact grain and relationship readiness,
- distinguish a star-schema source from a denormalized report source,
- prepare the Power BI connection checklist for Day 2,
- decide when Import, DirectQuery or Composite mode makes sense.


In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active catalog context for this session.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Pre-check

Verify that Workshop 1 has been completed. This notebook queries `gold.fact_sales_dashboard` and the dimension tables built in Workshop 1.

In [ ]:
if not spark.catalog.tableExists(f"{GOLD}.fact_sales_dashboard"):
    raise Exception("Pre-check failed. Run Workshop 1 first.")
print("[OK] Gold star schema found — ready to review")

## 1. What We Built Today

All Gold objects created during Day 1 (setup demo + Workshop 1):

In [ ]:
%sql
SHOW TABLES IN gold

Inspect the physical storage details of the main denormalized fact table:

In [ ]:
%sql
DESCRIBE DETAIL gold.fact_sales_dashboard

Row counts across all 5 Gold objects:

In [ ]:
%sql
SELECT 'dim_date'              AS object, COUNT(*) AS rows FROM gold.dim_date
UNION ALL SELECT 'dim_customer',           COUNT(*) FROM gold.dim_customer
UNION ALL SELECT 'dim_product',            COUNT(*) FROM gold.dim_product
UNION ALL SELECT 'fact_sales',             COUNT(*) FROM gold.fact_sales
UNION ALL SELECT 'fact_sales_dashboard',   COUNT(*) FROM gold.fact_sales_dashboard

Star schema structure:

```
dim_date ──┐
dim_customer ──┤── fact_sales ──► fact_sales_dashboard (denormalized, flat)
dim_product ──┘
```

`fact_sales_dashboard` contains all dimension attributes. Power BI can work with just this one table.

### Power BI developer pain points solved by Gold

| Pain in Power BI | Gold-layer response in Databricks |
|---|---|
| DAX measures copied across reports | shared KPI table or metric contract |
| Power Query performs joins and cleanup | joins and filters move upstream to Gold |
| DirectQuery visuals scan line-level detail | aggregate tables for summary pages |
| blank slicers from broken joins | relationship readiness checks before handoff |
| unclear ownership of revenue logic | table comments, KPI dictionary and decision log |

Experienced Power BI developers do not need Databricks to replace the semantic
model. They need Databricks to provide a trustworthy, governed source for it.


## 2. Gold as a BI Contract

A BI contract defines: (1) **grain** — what does one row represent? (2) **filter** — which orders are included? (3) **KPI owner** — who validates the definition? (4) **refresh SLA** — how stale is acceptable?

Verify the grain of `fact_sales_dashboard` — confirm one row per order line with no duplicates:

In [ ]:
%sql
SELECT
  COUNT(*)                AS total_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicates
FROM gold.fact_sales_dashboard

If `duplicates = 0`: grain is correct — one row per order line.

Tomorrow you will add the KPI layer (Workshop 2 at start of Day 2): `kpi_daily`, `kpi_monthly`, monthly aggregate table, and an incremental view.

### Relationship readiness and blank slicer check

A correct grain is not enough. Power BI relationships and slicers need complete
dimension attributes. Nulls in `customer_region`, `category` or `year_month`
usually mean an upstream join or dimension issue.


In [ ]:
%sql
SELECT
  SUM(CASE WHEN customer_region IS NULL THEN 1 ELSE 0 END) AS missing_customer_region,
  SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS missing_category,
  SUM(CASE WHEN year_month IS NULL THEN 1 ELSE 0 END) AS missing_year_month
FROM gold.fact_sales_dashboard


### Source choice for Day 2 Power BI demo

| Power BI need | Recommended Databricks object |
|---|---|
| detailed drill-through | `gold.fact_sales_dashboard` |
| star schema relationships | `gold.fact_sales` + dimensions |
| KPI cards and trends | Day 2 KPI tables from Workshop 2 |
| fast summary page | Day 2 monthly aggregate |
| live operational slice | Day 2 incremental view |

Day 1 stops at the core model. Day 2 adds the KPI and Power BI-specific serving
objects.


### Power BI handoff checklist

Before opening Power BI Desktop, participants should capture:

| Item | Where to find it |
|---|---|
| Catalog | configuration cell in every notebook |
| SQL Warehouse hostname | SQL Warehouses -> Connection details |
| HTTP path | SQL Warehouses -> Connection details |
| Authentication | PAT token or Entra ID depending on workspace setup |
| Primary Day 1 table | `gold.fact_sales_dashboard` |
| Required Day 2 tables | built by `w2_kpi_bi_readiness.ipynb` |

The point is to start Day 2 with a clean BI contract, not with a hunt through
Silver tables.


## 3. Tomorrow — Day 2 Plan

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

| Mode | Freshness | Best for |
|---|---|---|
| Import | Scheduled refresh (e.g., hourly) | Aggregate tables, KPI cards — small, fast |
| DirectQuery | Always live | Large detail tables — avoid for small teams |
| Composite | Mix per table | Best of both — recommended for this model |

**Day 2 run order:**
1. `Workshop 2` — add kpi_daily, kpi_monthly, monthly aggregate, incremental view
2. `day2_01` — Power BI connector setup, Import model, relationships, measures
3. `day2_02` — Performance, DABs, CI/CD (demo)
4. `day2_03` — AI/BI Dashboards & Genie (demo)

Before opening Power BI Desktop tomorrow:
- [ ] Note your catalog name: see CATALOG above
- [ ] You will need a PAT token (Personal Access Token) for the Databricks connector
- [ ] SQL Warehouse HTTP path: SQL Warehouses → your warehouse → Connection details
- [ ] Workshop 2 must finish before connecting Power BI